In [1]:
# imports, seed, device
import os, json, math, random, csv
from pathlib import Path
import torch, torch.nn as nn, torch.optim as optim
import torchvision.transforms as T
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, random_split, Subset
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path(".")
ARTS = ROOT / "artifacts"
FIGS = ARTS / "figures"
for p in (ROOT, ARTS, FIGS):
    p.mkdir(parents=True, exist_ok=True)

c:\Users\borga\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\cuda\__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
# dataset selection and dataloaders
# CIFAR10
DATASET = "CIFAR10"

num_classes = 10
transform = T.Compose([T.ToTensor(), T.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))])
DatasetClass = datasets.CIFAR10
dataset_args = {"root": "./data", "train": True, "download": True, "transform": transform}
test_args = {"root": "./data", "train": False, "download": True, "transform": transform}

full_train = DatasetClass(**dataset_args)
test_set = DatasetClass(**test_args)

train_val_split = 0.8
total = len(full_train)
n_train = int(total * train_val_split)
n_val = total - n_train
g = torch.Generator()
g.manual_seed(SEED)
train_set, val_set = random_split(full_train, [n_train, n_val], generator=g)

BATCH = 128
train_loader = DataLoader(train_set, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

In [3]:
# sanity-check: shapes, dtype
batch = next(iter(train_loader))
x, y = batch
print("x.shape", x.shape, "y.shape", y.shape, "dtype", x.dtype)
print("x min/max", float(x.min()), float(x.max()))
print("y unique (sample)", torch.unique(y)[:20])

x.shape torch.Size([128, 3, 32, 32]) y.shape torch.Size([128]) dtype torch.float32
x min/max -1.0 1.0
y unique (sample) tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


In [4]:
# model factory: MLP with optional BatchNorm and Dropout
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_sizes, num_classes, activation=nn.ReLU, dropout=0.0, use_bn=False):
        super().__init__()
        layers = []
        in_dim = input_dim
        for hs in hidden_sizes:
            layers.append(nn.Linear(in_dim, hs))
            if use_bn:
                layers.append(nn.BatchNorm1d(hs))
            layers.append(activation())
            if dropout and dropout>0:
                layers.append(nn.Dropout(dropout))
            in_dim = hs
        layers.append(nn.Linear(in_dim, num_classes))
        self.net = nn.Sequential(nn.Flatten(), *layers)
    def forward(self,x):
        return self.net(x)

def build_model(cfg):
    sample = full_train[0][0]
    input_dim = int(np.prod(sample.shape))
    return MLP(input_dim, cfg["hidden_sizes"], num_classes, nn.ReLU, cfg.get("dropout",0.0), cfg.get("batchnorm",False)).to(DEVICE)

In [5]:
# metrics and training/eval functions
def accuracy(logits, targets):
    return (logits.argmax(dim=1) == targets).float().mean().item()

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    running_acc = 0.0
    n = 0
    for xb,yb in loader:
        xb,yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        bs = xb.size(0)
        running_loss += loss.item() * bs
        running_acc += accuracy(logits, yb) * bs
        n += bs
    return running_loss / n, running_acc / n

def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    running_acc = 0.0
    n = 0
    with torch.no_grad():
        for xb,yb in loader:
            xb,yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            bs = xb.size(0)
            running_loss += loss.item() * bs
            running_acc += accuracy(logits, yb) * bs
            n += bs
    return running_loss / n, running_acc / n

In [6]:
# training loop with optional early stopping
def fit(model, train_loader, val_loader, cfg, save_best_path=None, early_stopping=None):
    criterion = nn.CrossEntropyLoss()
    opt_cfg = cfg["optimizer"]
    if opt_cfg["type"].lower() == "adam":
        optimizer = optim.Adam(model.parameters(), lr=opt_cfg["lr"], weight_decay=opt_cfg.get("weight_decay",0.0))
    else:
        optimizer = optim.SGD(model.parameters(), lr=opt_cfg["lr"], momentum=opt_cfg.get("momentum",0.0), weight_decay=opt_cfg.get("weight_decay",0.0))
    epochs = cfg["epochs"]
    history = {"train_loss":[], "train_acc":[], "val_loss":[], "val_acc":[]}
    best_val = float("inf")
    best_acc = 0.0
    best_state = None
    patience = early_stopping.get("patience") if early_stopping else None
    wait = 0
    for ep in range(1, epochs+1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
        history["val_loss"].append(val_loss); history["val_acc"].append(val_acc)
        if val_acc > best_acc:
            best_acc = val_acc
            best_val = val_loss
            best_state = {k:v.cpu() for k,v in model.state_dict().items()}
            wait = 0
        else:
            if patience is not None:
                wait += 1
                if wait >= patience:
                    break
    if save_best_path and best_state is not None:
        torch.save(best_state, save_best_path)
    return history, best_acc, best_val, ep

In [7]:
# runner that records runs
import pandas as pd
runs = []

def run_experiment(exp_id, model_cfg, train_loader, val_loader, cfg_train, save_best=False):
    model = build_model(model_cfg)
    save_path = None
    if save_best:
        save_path = ARTS / "best_model.pt"
    hist, best_acc, best_loss, epochs_trained = fit(model, train_loader, val_loader, cfg_train, save_best_path=save_path, early_stopping=cfg_train.get("early_stopping"))
    runs.append({
        "experiment_id": exp_id,
        "dataset": DATASET,
        "seed": SEED,
        "model_summary": f'hidden={model_cfg["hidden_sizes"]}_dropout={model_cfg.get("dropout",0)}_bn={model_cfg.get("batchnorm",False)}',
        "optimizer": cfg_train["optimizer"]["type"],
        "lr": cfg_train["optimizer"]["lr"],
        "momentum": cfg_train["optimizer"].get("momentum",0.0),
        "weight_decay": cfg_train["optimizer"].get("weight_decay",0.0),
        "epochs_trained": epochs_trained,
        "best_val_accuracy": best_acc,
        "best_val_loss": best_loss
    })
    return hist, model, best_acc

In [8]:
# prepare model configs and training configs for E1-E4
base_model_cfg = {"hidden_sizes":[512,256,128], "dropout":0.0, "batchnorm":False}
EPOCHS = 10

E1_model = dict(base_model_cfg)
E2_model = dict(base_model_cfg); E2_model["dropout"] = 0.3
E3_model = dict(base_model_cfg); E3_model["batchnorm"] = True

adam_cfg = {"epochs": EPOCHS, "optimizer": {"type":"Adam","lr":1e-3, "weight_decay":0.0}}

# run E1-E3
hist_E1, _, acc1 = run_experiment("E1", E1_model, train_loader, val_loader, adam_cfg, save_best=False)
hist_E2, _, acc2 = run_experiment("E2", E2_model, train_loader, val_loader, adam_cfg, save_best=False)
hist_E3, _, acc3 = run_experiment("E3", E3_model, train_loader, val_loader, adam_cfg, save_best=False)

# pick best between E2 and E3
best_choice = "E2" if acc2 >= acc3 else "E3"
best_model_cfg = E2_model if best_choice=="E2" else E3_model

# E4: train best with early stopping and save best model and config
adam_cfg_es = dict(adam_cfg); adam_cfg_es["early_stopping"] = {"patience":4}
hist_E4, best_model, best_acc_E4 = run_experiment("E4", best_model_cfg, train_loader, val_loader, adam_cfg_es, save_best=True)
best_config = {"experiment":"E4","chosen":best_choice,"model_cfg":best_model_cfg,"train_cfg":adam_cfg_es,"seed":SEED,"dataset":DATASET}
with open(ARTS / "best_config.json","w") as f:
    json.dump(best_config, f, indent=2)

In [9]:
# O1-O3 experiments (use architecture from best_model_cfg)
# O1: Adam lr too large
o_epochs = 6
O1_cfg = {"epochs": o_epochs, "optimizer": {"type":"Adam","lr":1e-1, "weight_decay":0.0}}
hist_O1, _, _ = run_experiment("O1", best_model_cfg, train_loader, val_loader, O1_cfg, save_best=False)

# O2: Adam lr too small
O2_cfg = {"epochs": o_epochs, "optimizer": {"type":"Adam","lr":1e-5, "weight_decay":0.0}}
hist_O2, _, _ = run_experiment("O2", best_model_cfg, train_loader, val_loader, O2_cfg, save_best=False)

# O3: SGD+momentum + weight decay
O3_cfg = {"epochs": 12, "optimizer": {"type":"SGD","lr":1e-2, "momentum":0.9, "weight_decay":1e-4}}
hist_O3, _, _ = run_experiment("O3", best_model_cfg, train_loader, val_loader, O3_cfg, save_best=False)

In [10]:
# save runs.csv
df = pd.DataFrame(runs)
df.to_csv(ARTS / "runs.csv", index=False)

In [11]:
# plotting helpers (save but user may replace after running)
def plot_curves(history, title, path):
    plt.figure(figsize=(6,4))
    plt.plot(history["train_loss"], label="train_loss")
    plt.plot(history["val_loss"], label="val_loss")
    plt.title(title)
    plt.xlabel("epoch")
    plt.legend()
    plt.tight_layout()
    plt.savefig(path)
    plt.close()

plot_curves(hist_E4, "E4 (best) loss", FIGS / "curves_best.png")
# LR extremes combined
plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.plot(hist_O1["train_loss"], label="O1 train_loss"); plt.plot(hist_O1["val_loss"], label="O1 val_loss"); plt.title("O1 large lr"); plt.legend()
plt.subplot(1,2,2)
plt.plot(hist_O2["train_loss"], label="O2 train_loss"); plt.plot(hist_O2["val_loss"], label="O2 val_loss"); plt.title("O2 small lr"); plt.legend()
plt.tight_layout(); plt.savefig(FIGS / "curves_lr_extremes.png"); plt.close()

In [12]:
# final test evaluation of saved best model (load best_model.pt once and eval on test)
best_state = torch.load(ARTS / "best_model.pt", map_location="cpu")
model_for_test = build_model(best_model_cfg)
model_for_test.load_state_dict(best_state)
test_loss, test_acc = evaluate(model_for_test.to(DEVICE), test_loader, nn.CrossEntropyLoss())
with open(ARTS / "test_result.txt","w") as f:
    f.write(f"test_loss: {test_loss}\ntest_acc: {test_acc}\n")
print("Test:", test_loss, test_acc)

Test: 1.4107696380615233 0.5419
